In [ ]:
%sql
-- silver and gold schemas
USE CATALOG energy;
CREATE SCHEMA IF NOT EXISTS d2_silver;
CREATE SCHEMA IF NOT EXISTS d2_gold;

In [ ]:
%sql
-- DIM ZONE
CREATE TABLE IF NOT EXISTS d2_gold.dim_zone (
  zone_code STRING NOT NULL,
  zone_label STRING,
  country STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  is_cwe BOOLEAN COMMENT 'Central Western Europe: BE, FR, NL, DE-LU',
  CONSTRAINT dim_zone_pk PRIMARY KEY (zone_code) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d2_gold.dim_zone VALUES
  ('BE', 'Belgium', 'Belgium', 50.64, 4.67, true),
  ('FR', 'France', 'France', 46.60, 2.40, true),
  ('NL', 'Netherlands', 'Netherlands', 52.20, 5.50, true),
  ('DE_LU', 'Germany-Luxembourg', 'Germany', 51.10, 10.40, true),
  ('AT', 'Austria', 'Austria', 47.60, 14.10, false),
  ('CH', 'Switzerland', 'Switzerland', 46.80, 8.20, false),
  ('CZ', 'Czechia', 'Czechia', 49.80, 15.50, false),
  ('PL', 'Poland', 'Poland', 52.10, 19.40, false),
  ('DK_1', 'Denmark West', 'Denmark', 56.20, 9.00, false),
  ('ES', 'Spain', 'Spain', 40.20, -3.70, false);

In [ ]:
%sql
-- DIM BORDER (one row per border, A-B)
CREATE TABLE IF NOT EXISTS d2_gold.dim_border (
  border_key STRING NOT NULL,
  border_label STRING,
  zone_a STRING,
  zone_b STRING,
  zone_a_label STRING,
  zone_b_label STRING,
  involves_be BOOLEAN,
  is_cwe_internal BOOLEAN,
  CONSTRAINT dim_border_pk PRIMARY KEY (border_key) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d2_gold.dim_border
WITH borders (border_key, zone_a, zone_b) AS (VALUES
  ('BE-FR', 'BE', 'FR'),
  ('BE-NL', 'BE', 'NL'),
  ('BE-DE_LU', 'BE', 'DE_LU'),
  ('FR-DE_LU', 'FR', 'DE_LU'),
  ('FR-CH', 'FR', 'CH'),
  ('FR-ES', 'FR', 'ES'),
  ('NL-DE_LU', 'NL', 'DE_LU'),
  ('NL-DK_1', 'NL', 'DK_1'),
  ('DE_LU-AT', 'DE_LU', 'AT'),
  ('DE_LU-CH', 'DE_LU', 'CH'),
  ('DE_LU-CZ', 'DE_LU', 'CZ'),
  ('DE_LU-PL', 'DE_LU', 'PL'),
  ('DE_LU-DK_1', 'DE_LU', 'DK_1'),
  ('AT-CH', 'AT', 'CH'),
  ('AT-CZ', 'AT', 'CZ'),
  ('CZ-PL', 'CZ', 'PL')
)
SELECT
  b.border_key,
  concat(za.zone_label, ' - ', zb.zone_label) AS border_label,
  b.zone_a,
  b.zone_b,
  za.zone_label AS zone_a_label,
  zb.zone_label AS zone_b_label,
  b.zone_a = 'BE' OR b.zone_b = 'BE' AS involves_be,
  za.is_cwe AND zb.is_cwe AS is_cwe_internal
FROM borders b
JOIN d2_gold.dim_zone za ON b.zone_a = za.zone_code
JOIN d2_gold.dim_zone zb ON b.zone_b = zb.zone_code;

In [ ]:
%sql
-- DIM ROUTE (each border in both directions, with coordinates, for the flow map)
CREATE TABLE IF NOT EXISTS d2_gold.dim_route (
  route_key STRING NOT NULL,
  route_label STRING,
  border_key STRING,
  from_zone STRING,
  to_zone STRING,
  from_label STRING,
  to_label STRING,
  from_lat DOUBLE,
  from_lon DOUBLE,
  to_lat DOUBLE,
  to_lon DOUBLE,
  involves_be BOOLEAN,
  CONSTRAINT dim_route_pk PRIMARY KEY (route_key) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d2_gold.dim_route
WITH routes AS (
  SELECT border_key, zone_a AS from_zone, zone_b AS to_zone FROM d2_gold.dim_border
  UNION ALL
  SELECT border_key, zone_b AS from_zone, zone_a AS to_zone FROM d2_gold.dim_border
)
SELECT
  concat(r.from_zone, '>', r.to_zone) AS route_key,
  concat(zf.zone_label, ' -> ', zt.zone_label) AS route_label,
  r.border_key,
  r.from_zone,
  r.to_zone,
  zf.zone_label AS from_label,
  zt.zone_label AS to_label,
  zf.latitude AS from_lat,
  zf.longitude AS from_lon,
  zt.latitude AS to_lat,
  zt.longitude AS to_lon,
  r.from_zone = 'BE' OR r.to_zone = 'BE' AS involves_be
FROM routes r
JOIN d2_gold.dim_zone zf ON r.from_zone = zf.zone_code
JOIN d2_gold.dim_zone zt ON r.to_zone = zt.zone_code;

In [ ]:
#DIM DATE
from pyspark.sql.functions import col, date_format, year, quarter, month, dayofweek, weekofyear, when, concat, lit

dates_df = spark.sql("""
  SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) AS date
""")

# labels are prefixed ('01 - Jan', '1 - Winter') so Power BI sorts them in the right order
dim_date_df = dates_df \
    .withColumn("date_key", date_format("date", "yyyyMMdd").cast("int")) \
    .withColumn("year", year("date")) \
    .withColumn("quarter_label", concat(lit("Q"), quarter("date"))) \
    .withColumn("month_num", month("date")) \
    .withColumn("month_label", concat(date_format("date", "MM"), lit(" - "), date_format("date", "MMM"))) \
    .withColumn("year_month", date_format("date", "yyyy-MM")) \
    .withColumn("iso_week", weekofyear("date")) \
    .withColumn("day_of_week_num", (dayofweek("date") + 5) % 7 + 1) \
    .withColumn("day_label", concat(col("day_of_week_num"), lit(" - "), date_format("date", "E"))) \
    .withColumn("is_weekend", col("day_of_week_num") >= 6) \
    .withColumn("season_label",
        when(col("month_num").isin(12, 1, 2), "1 - Winter")
        .when(col("month_num").isin(3, 4, 5), "2 - Spring")
        .when(col("month_num").isin(6, 7, 8), "3 - Summer")
        .otherwise("4 - Autumn"))

dim_date_df.createOrReplaceTempView("v_dim_date")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS d2_gold.dim_date (
  date_key INT NOT NULL,
  date DATE,
  year INT,
  quarter_label STRING,
  month_num INT,
  month_label STRING,
  year_month STRING,
  iso_week INT,
  day_of_week_num INT,
  day_label STRING,
  is_weekend BOOLEAN,
  season_label STRING,
  CONSTRAINT dim_date_pk PRIMARY KEY (date_key) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d2_gold.dim_date
SELECT date_key, date, year, quarter_label, month_num, month_label, year_month,
       iso_week, day_of_week_num, day_label, is_weekend, season_label
FROM v_dim_date;

In [ ]:
%sql
-- DIM HOUR
CREATE TABLE IF NOT EXISTS d2_gold.dim_hour (
  hour_key INT NOT NULL,
  hour_label STRING,
  day_period STRING,
  is_peak BOOLEAN,
  CONSTRAINT dim_hour_pk PRIMARY KEY (hour_key) NOT ENFORCED
) USING DELTA;

INSERT OVERWRITE d2_gold.dim_hour
SELECT
  CAST(id AS INT) AS hour_key,
  concat(lpad(CAST(id AS STRING), 2, '0'), ':00') AS hour_label,
  CASE
    WHEN id < 6 OR id >= 22 THEN '1 - Night'
    WHEN id < 10 THEN '2 - Morning'
    WHEN id < 17 THEN '3 - Daytime'
    ELSE '4 - Evening'
  END AS day_period,
  id BETWEEN 8 AND 19 AS is_peak
FROM range(24);

In [ ]:
%sql
-- FACT TABLES
CREATE TABLE IF NOT EXISTS d2_gold.fact_price_hourly (
  ts_utc TIMESTAMP NOT NULL,
  datetime_local TIMESTAMP,
  date_key INT,
  hour_key INT,
  zone_code STRING,
  price_eur_mwh DOUBLE,
  net_position_mwh DOUBLE,
  is_negative_price BOOLEAN,
  CONSTRAINT fk_price_date FOREIGN KEY (date_key) REFERENCES d2_gold.dim_date(date_key) NOT ENFORCED,
  CONSTRAINT fk_price_hour FOREIGN KEY (hour_key) REFERENCES d2_gold.dim_hour(hour_key) NOT ENFORCED,
  CONSTRAINT fk_price_zone FOREIGN KEY (zone_code) REFERENCES d2_gold.dim_zone(zone_code) NOT ENFORCED
) USING DELTA;

CREATE TABLE IF NOT EXISTS d2_gold.fact_border_hourly (
  ts_utc TIMESTAMP NOT NULL,
  datetime_local TIMESTAMP,
  date_key INT,
  hour_key INT,
  border_key STRING,
  price_a_eur_mwh DOUBLE,
  price_b_eur_mwh DOUBLE,
  spread_a_b_eur_mwh DOUBLE,
  abs_spread_eur_mwh DOUBLE,
  is_converged BOOLEAN,
  is_congested BOOLEAN,
  flow_a_to_b_mwh DOUBLE,
  flow_b_to_a_mwh DOUBLE,
  net_flow_a_to_b_mwh DOUBLE,
  sched_a_to_b_mwh DOUBLE,
  sched_b_to_a_mwh DOUBLE,
  net_sched_a_to_b_mwh DOUBLE,
  congestion_rent_eur DOUBLE,
  is_adverse_flow BOOLEAN,
  CONSTRAINT fk_border_date FOREIGN KEY (date_key) REFERENCES d2_gold.dim_date(date_key) NOT ENFORCED,
  CONSTRAINT fk_border_hour FOREIGN KEY (hour_key) REFERENCES d2_gold.dim_hour(hour_key) NOT ENFORCED,
  CONSTRAINT fk_border_border FOREIGN KEY (border_key) REFERENCES d2_gold.dim_border(border_key) NOT ENFORCED
) USING DELTA;

CREATE TABLE IF NOT EXISTS d2_gold.fact_route_flow_hourly (
  ts_utc TIMESTAMP NOT NULL,
  datetime_local TIMESTAMP,
  date_key INT,
  hour_key INT,
  route_key STRING,
  flow_mwh DOUBLE,
  reverse_flow_mwh DOUBLE,
  net_flow_mwh DOUBLE,
  sched_mwh DOUBLE,
  reverse_sched_mwh DOUBLE,
  net_sched_mwh DOUBLE,
  CONSTRAINT fk_route_date FOREIGN KEY (date_key) REFERENCES d2_gold.dim_date(date_key) NOT ENFORCED,
  CONSTRAINT fk_route_hour FOREIGN KEY (hour_key) REFERENCES d2_gold.dim_hour(hour_key) NOT ENFORCED,
  CONSTRAINT fk_route_route FOREIGN KEY (route_key) REFERENCES d2_gold.dim_route(route_key) NOT ENFORCED
) USING DELTA;

CREATE TABLE IF NOT EXISTS d2_gold.fact_region_hourly (
  ts_utc TIMESTAMP NOT NULL,
  region_code STRING,
  price_min_eur_mwh DOUBLE,
  price_max_eur_mwh DOUBLE,
  max_spread_eur_mwh DOUBLE,
  n_price_groups INT,
  is_full_convergence BOOLEAN,
  be_price_eur_mwh DOUBLE,
  be_is_highest BOOLEAN,
  be_is_lowest BOOLEAN,
  datetime_local TIMESTAMP,
  date_key INT,
  hour_key INT,
  CONSTRAINT fk_region_date FOREIGN KEY (date_key) REFERENCES d2_gold.dim_date(date_key) NOT ENFORCED,
  CONSTRAINT fk_region_hour FOREIGN KEY (hour_key) REFERENCES d2_gold.dim_hour(hour_key) NOT ENFORCED
) USING DELTA;